# Lekce 02 - Prozkoumání Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** je jednotné prostředí pro vytváření AI agentů. Poskytuje čistou, složitelnou architekturu se čtyřmi hlavními stavebními bloky:

- **Client** – připojuje se k AI modelovému endpointu a zajišťuje komunikaci
- **Agent** – obaluje klienta s instrukcemi a definicemi nástrojů
- **Tools** – rozšiřují schopnosti agenta o vlastní funkce, které může model volat
- **Session** – uchovává historii konverzace pro vícekrokové interakce

V této lekci vytvoříme **agenta pro rezervaci cestování**, který pomocí těchto konceptů kontroluje dostupnost destinací.


## Nastavení


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Pochopení architektury Agent Frameworku

Microsoft Agent Framework následuje vícevrstvou architekturu:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – `FoundryChatClient` se připojuje k nasazení Azure OpenAI. Zajišťuje autentifikaci, formátování požadavků a parsování odpovědí.
2. **Agent** – Vytvořený z klienta přes `provider.create_agent()`, agent kombinuje přístup k modelu s instrukcemi (systémový prompt) a nástroji.
3. **Nástroje** – Python funkce dekorované pomocí `@tool`, které může agent volat k vykonání akcí nebo získání dat.
4. **Relace** – Objekt `AgentSession` (vytvořený přes `agent.create_session()`), který ukládá historii konverzace, umožňující vícekolovou dialogickou interakci, kdy si agent pamatuje předchozí kontext.

Pojďme si každý stupeň vybudovat krok za krokem.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Přidávání nástrojů pomocí dekorátoru @tool

Nástroje umožňují agentům provádět akce nad rámec generování textu. Dekorátor `@tool` přemění běžnou Python funkci na něco, co agent může zavolat.

Hlavní body:
- Použijte `Annotated[type, "popis"]` aby model rozuměl každému parametru.
- Dokumentační řetězec se stane popisem nástroje, který model vidí.
- `approval_mode="never_require"` znamená, že nástroj běží automaticky bez potvrzení uživatele.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Vytvoření agenta s nástroji

Nyní kombinujeme klienta, instrukce a nástroje do agenta. `instructions` fungují jako systémový prompt — definují osobnost a chování agenta.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Vícekolové konverzace se relacemi

`AgentSession` (vytvořená pomocí `agent.create_session()`) sleduje všechny zprávy v konverzaci. Předáváním stejné relace do každého volání `agent.run()` má agent přístup k celé historii konverzace a může odkazovat na dřívější zprávy.

Předáváme `tools=[check_destination_availability]`, aby agent mohl během každého kola volat náš kontrolor dostupnosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Shrnutí

V této lekci jste prozkoumali čtyři pilíře Microsoft Agent Framework:

| Koncept | Co jste se naučili |
|---------|------------------|
| **Klient** | `FoundryChatClient` se připojuje k Azure OpenAI pomocí autentizace na základě přihlašovacích údajů |
| **Agent** | `provider.create_agent()` propojuje připojení modelu s instrukcemi a názvem |
| **Nástroje** | Dekorátor `@tool` zpřístupňuje Python funkce, které agent může volat |
| **Relace** | `agent.create_session()` udržuje historii konverzace přes více kol |

Tyto stavební bloky se složí dohromady k vytvoření agentů, kteří mohou vést přirozené konverzace, volat externí funkce a udržovat kontext — základ pro pokročilejší vzory agentů v pozdějších lekcích.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
